# Лабораторная работа 8. Кластеризация и EM-алгоритм

**Курс «Машинное обучение», 4 курс, каф. ФН1**

| | |
|---|---|
| Место в курсе | после лекции 7 |
| Опора на лекции | лекция 7: функционал качества кластеризации и метод $K$-средних (опр. 7.1), смеси распределений (опр. 7.4), EM-алгоритм (опр. 7.6) и его монотонность (теорема 7.7), ответственности (опр. 7.8), иерархическая кластеризация и linkage (опр. 7.11) |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Первое занятие курса без учителя. Понять, **что именно** оптимизирует каждый метод — и почему из этого сразу следует, какие структуры данных он найти не сможет. Осознать, что «правильного» числа кластеров не существует, и научиться выбирать его осмысленно.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab08_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Индивидуальный вариант.** Датасет и набор методов выдаются по вашему ФИО
> (см. ячейку ниже) — и на семинаре, и в домашней работе.

Кластеризация — задача **обучения без учителя**: ответов $y_i$ нет. Это меняет
всё, к чему мы привыкли: нет ни эмпирического риска относительно истинных
ответов, ни скользящего контроля в прежнем смысле, ни однозначного критерия
правильности. Поэтому здесь особенно важно понимать устройство метода.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans, SpectralClustering
from sklearn.metrics import (adjusted_mutual_info_score, adjusted_rand_score,
                             silhouette_samples, silhouette_score)
from sklearn.mixture import GaussianMixture
from sklearn.datasets import make_blobs, make_circles
from sklearn.preprocessing import StandardScaler
from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=8)
describe_variant(variant)

---
# Часть 1. $K$-средних и его функционал

Определение 7.1: минимизируется сумма квадратов расстояний до центров

$$
Q(C, \mu) = \sum_{k=1}^{K}\sum_{i \in C_k}\|x_i - \mu_k\|^2 \;\longrightarrow\; \min .
$$

Алгоритм Ллойда чередует два шага, каждый из которых решает свою подзадачу
**точно**: отнести объект к ближайшему центру и пересчитать центры как средние.
Поэтому $Q$ не возрастает, и метод сходится за конечное число шагов — но
только к локальному минимуму.

In [ ]:
X_b, _ = make_blobs(n_samples=500, centers=4, cluster_std=1.1, random_state=RANDOM_STATE)
X_b = StandardScaler().fit_transform(X_b)

km = KMeans(n_clusters=4, n_init=1, init="random", random_state=RANDOM_STATE).fit(X_b)
print(f"Q (инерция) при одном запуске со случайной инициализацией: {km.inertia_:.4f}")
print(f"Q при n_init=10 и k-means++:                              "
      f"{KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE).fit(X_b).inertia_:.4f}")

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

res = {"случайная": [], "k-means++": []}
# TODO (3-4 строки): по 60 запусков KMeans(4, n_init=1) с init="random"
#   и init="k-means++" (меняйте random_state), собирайте .inertia_.
#   Затем для каждой инициализации выведите лучшее Q, среднее Q и долю
#   запусков, попавших в лучший минимум (в пределах 1 %).

In [ ]:
fig, ax = plt.subplots()
ax.hist([res["случайная"], res["k-means++"]], bins=25, label=["случайная", "k-means++"])
ax.set_xlabel("итоговое $Q$"); ax.set_ylabel("число запусков"); ax.legend()
ax.set_title("60 запусков: качество локального минимума")
plt.tight_layout(); plt.show()

> **Вывод.** Какая инициализация чаще попадает в хороший минимум и почему? Зачем в `sklearn` параметр `n_init`?
>
> *(ваш ответ здесь)*

---
# Часть 2. Чего $K$-средних не умеет — и почему

Функционал $\sum_k\sum_{i\in C_k}\|x_i-\mu_k\|^2$ жёстко задаёт форму
кластеров. Разбиение по ближайшему центру — это **диаграмма Вороного**, то есть
кластеры всегда выпуклые и разделены гиперплоскостями. Кроме того, квадрат
расстояния штрафует крупные кластеры сильнее мелких.

Трудный случай берётся из вашего варианта: `variant["hard_case"]`.

In [ ]:
def make_hard(case, n=600, seed=RANDOM_STATE):
    g = np.random.default_rng(seed)
    if case.startswith("кластеры разного размера"):
        X = np.vstack([g.normal([0, 0], 0.35, (n // 10, 2)),
                       g.normal([3, 0], 0.35, (n // 10, 2)),
                       g.normal([1.5, 3.5], 1.6, (8 * n // 10, 2))])
        y = np.r_[np.zeros(n // 10), np.ones(n // 10), np.full(8 * n // 10, 2)]
    elif case.startswith("анизотропные"):
        X, y = make_blobs(n_samples=n, centers=3, cluster_std=0.9, random_state=seed)
        X = X @ np.array([[0.6, -0.63], [-0.41, 0.85]])       # вытягиваем кластеры
    elif case.startswith("кластеры разной плотности"):
        X = np.vstack([g.normal([0, 0], 0.25, (n // 3, 2)),
                       g.normal([2.5, 0], 0.9, (n // 3, 2)),
                       g.normal([1.2, 3.0], 1.8, (n // 3, 2))])
        y = np.repeat([0, 1, 2], n // 3)
    else:                                                     # вложенные кольца
        X, y = make_circles(n_samples=n, noise=0.06, factor=0.45, random_state=seed)
    return StandardScaler().fit_transform(X), np.asarray(y, int)


case = variant["hard_case"]
X_h, y_h = make_hard(case)
K_true = len(np.unique(y_h))
print(f"трудный случай вашего варианта: {case} (истинных кластеров {K_true})")

In [ ]:
methods = {
    "K-средних": lambda X: KMeans(K_true, n_init=10, random_state=0).fit_predict(X),
    "иерархическая (ward)": lambda X: AgglomerativeClustering(K_true, linkage="ward").fit_predict(X),
    "иерархическая (single)": lambda X: AgglomerativeClustering(K_true, linkage="single").fit_predict(X),
    "DBSCAN": lambda X: DBSCAN(eps=0.3, min_samples=8).fit_predict(X),
    "спектральная": lambda X: SpectralClustering(K_true, affinity="nearest_neighbors",
                                                 random_state=0).fit_predict(X),
}

fig, axes = plt.subplots(1, len(methods) + 1, figsize=(3.3 * (len(methods) + 1), 3.6))
axes[0].scatter(*X_h.T, c=y_h, s=10, cmap="viridis")
axes[0].set_title("истинные классы", fontsize=9); axes[0].set_xticks([]); axes[0].set_yticks([])
rows = []
for ax, (nm, fn) in zip(axes[1:], methods.items()):
    lab = fn(X_h)
    ax.scatter(*X_h.T, c=lab, s=10, cmap="viridis")
    ax.set_title(f"{nm}\nARI = {adjusted_rand_score(y_h, lab):.3f}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    rows.append({"метод": nm, "ARI": adjusted_rand_score(y_h, lab),
                 "AMI": adjusted_mutual_info_score(y_h, lab),
                 "кластеров найдено": len(set(lab) - {-1})})
plt.tight_layout(); plt.show()
display(pd.DataFrame(rows).set_index("метод").round(3))

> **Вывод.** Насколько плохо сработал $K$-средних на вашем случае и **почему именно** — какое свойство функционала это объясняет? Кто справился и за счёт чего?
>
> *(ваш ответ здесь)*

---
# Часть 3. Смеси распределений и EM

Определение 7.4: плотность смеси $p(x) = \sum_k w_k\,p_k(x)$. Прямая
максимизация правдоподобия сложна из-за логарифма суммы; EM (опр. 7.6)
вводит скрытые переменные и чередует:

**E-шаг** — ответственности (опр. 7.8):
$\gamma_{ik} = \dfrac{w_k\,p_k(x_i)}{\sum_s w_s\,p_s(x_i)}$;

**M-шаг** — пересчёт параметров как **взвешенных** оценок с весами $\gamma_{ik}$.

Теорема 7.7: логарифм правдоподобия **не убывает** на каждой итерации.
Проверим — это главный эксперимент занятия.

In [ ]:
X_g, _ = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=RANDOM_STATE)
X_g = X_g @ np.array([[0.6, -0.6], [-0.4, 0.8]])      # делаем кластеры вытянутыми
X_g = StandardScaler().fit_transform(X_g)

cov_type = variant["gmm_cov"]
print(f"тип ковариации вашего варианта: {cov_type}")

# warm_start + max_iter=1 позволяет проследить logL по шагам
gm = GaussianMixture(3, covariance_type=cov_type, max_iter=1, warm_start=True,
                     random_state=RANDOM_STATE, init_params="k-means++")
logL = []
for _ in range(60):
    gm.fit(X_g)
    logL.append(gm.score(X_g) * len(X_g))
logL = np.array(logL)

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

# logL -- массив значений логарифма правдоподобия по итерациям EM
# TODO (2-3 строки): посчитайте приросты за шаг (np.diff), выведите минимальный
#   и проверьте, что он неотрицателен -- это и есть теорема 7.7.

In [ ]:
gm_full = GaussianMixture(3, covariance_type=cov_type, n_init=5,
                          random_state=RANDOM_STATE).fit(X_g)
gamma = gm_full.predict_proba(X_g)
labels = gamma.argmax(axis=1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
ax1.plot(logL, "o-", lw=2, ms=3)
ax1.set_xlabel("итерация EM"); ax1.set_ylabel(r"$\ln L$")
ax1.set_title("Теорема 7.7: правдоподобие не убывает")

ax2.scatter(*X_g.T, c=labels, s=18, cmap="viridis", alpha=np.clip(gamma.max(1), .25, 1))
for k in range(3):
    cov = (gm_full.covariances_[k] if cov_type == "full" else
           np.diag(gm_full.covariances_[k]) if cov_type == "diag" else
           np.eye(2) * gm_full.covariances_[k])
    vals, vecs = np.linalg.eigh(cov)
    ang = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    for s in (1, 2):
        ax2.add_patch(plt.matplotlib.patches.Ellipse(
            gm_full.means_[k], *(2 * s * np.sqrt(vals[::-1])), angle=ang,
            fill=False, lw=1.6, color="black"))
ax2.set_title("Компоненты смеси (прозрачность = неуверенность)")
plt.tight_layout(); plt.show()

### $K$-средних как предельный случай EM

Возьмём сферические ковариации $\Sigma_k = \sigma^2 I$ и устремим $\sigma\to0$.
Ответственности
$\gamma_{ik} \propto w_k\exp\bigl(-\|x_i-\mu_k\|^2/(2\sigma^2)\bigr)$
превращаются в жёсткое отнесение к ближайшему центру, а M-шаг — в пересчёт
средних. То есть **$K$-средних — это EM для сферической смеси при $\sigma\to0$**.

In [ ]:
km3 = KMeans(3, n_init=10, random_state=RANDOM_STATE).fit(X_g)
d2 = ((X_g[:, None, :] - km3.cluster_centers_[None, :, :]) ** 2).sum(-1)

rows = []
for sigma in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02]:
    logg = -d2 / (2 * sigma ** 2)
    gam = np.exp(logg - logg.max(axis=1, keepdims=True))
    gam /= gam.sum(axis=1, keepdims=True)
    rows.append({"sigma": sigma, "средняя max-ответственность": gam.max(axis=1).mean(),
                 "доля объектов с gamma > 0.99": np.mean(gam.max(axis=1) > 0.99),
                 "ARI с K-средних": adjusted_rand_score(gam.argmax(1), km3.labels_)})
display(pd.DataFrame(rows).set_index("sigma").round(4))

> **Вывод.** Подтвердилась ли монотонность? Что происходит с ответственностями при $\sigma\to0$? Когда мягкое отнесение предпочтительнее жёсткого?
>
> *(ваш ответ здесь)*

---
# Часть 4. Сколько кластеров

Меток нет, поэтому «правильное» $K$ выбирается косвенно:

* **метод локтя**: $Q(K)$ убывает всегда, ищут точку излома;
* **силуэт**: $s_i = \dfrac{b_i - a_i}{\max(a_i, b_i)}$, где $a_i$ — среднее
  расстояние внутри своего кластера, $b_i$ — до ближайшего чужого;
* **информационные критерии** для смеси: $\mathrm{AIC} = -2\ln L + 2p$,
  $\mathrm{BIC} = -2\ln L + p\ln\ell$.

Ваш вариант: `variant["k_selection"]`. Проверим на данных с известным ответом.

In [ ]:
X_k, y_k = make_blobs(n_samples=600, centers=4, cluster_std=1.15, random_state=RANDOM_STATE)
X_k = StandardScaler().fit_transform(X_k)

rows = []
for K in range(2, 11):
    km = KMeans(K, n_init=10, random_state=RANDOM_STATE).fit(X_k)
    gm = GaussianMixture(K, covariance_type="full", random_state=RANDOM_STATE,
                         n_init=3).fit(X_k)
    rows.append({"K": K, "Q (инерция)": km.inertia_,
                 "силуэт": silhouette_score(X_k, km.labels_),
                 "AIC": gm.aic(X_k), "BIC": gm.bic(X_k),
                 "ARI с истиной": adjusted_rand_score(y_k, km.labels_)})
sel = pd.DataFrame(rows).set_index("K")
display(sel.round(3))

In [ ]:
choices = {
    "локоть (макс. вторая разность Q)": int(sel.index[1 + np.argmax(np.diff(sel["Q (инерция)"], 2))]),
    "силуэт": int(sel["силуэт"].idxmax()),
    "AIC": int(sel["AIC"].idxmin()), "BIC": int(sel["BIC"].idxmin()),
    "ARI (знание истины)": int(sel["ARI с истиной"].idxmax()),
}
print("критерии вашего варианта:", variant["k_selection"], "\n")
for k, v in choices.items():
    print(f"  {k:34s} -> K = {v}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(sel.index, sel["Q (инерция)"], "o-", lw=2)
axes[0].set_xlabel("$K$"); axes[0].set_ylabel("$Q$"); axes[0].set_title("Метод локтя")
axes[1].plot(sel.index, sel["силуэт"], "o-", lw=2, color="#C97A2B")
axes[1].axvline(choices["силуэт"], ls="--", color="black")
axes[1].set_xlabel("$K$"); axes[1].set_ylabel("средний силуэт"); axes[1].set_title("Силуэт")
axes[2].plot(sel.index, sel["AIC"], "o-", lw=2, label="AIC")
axes[2].plot(sel.index, sel["BIC"], "s-", lw=2, label="BIC")
axes[2].set_xlabel("$K$"); axes[2].set_title("Информационные критерии"); axes[2].legend()
plt.tight_layout(); plt.show()

> **Вывод.** Совпали ли ответы разных критериев с истинным $K=4$? Почему $Q(K)$ нельзя минимизировать напрямую?
>
> *(ваш ответ здесь)*

---
# Часть 5. Иерархическая кластеризация и эффект цепочки

Определение 7.11: агломеративный алгоритм начинает с $\ell$ одноэлементных
кластеров и на каждом шаге объединяет два ближайших. Способ измерения расстояния
между кластерами (linkage) определяет всё поведение: `single` берёт минимум по
парам, `complete` — максимум, `ward` — прирост $Q$ при слиянии.

Ваш вариант: `variant["linkage"]`.

In [ ]:
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage as scipy_linkage

X_d, y_d = make_blobs(n_samples=160, centers=3, cluster_std=1.0, random_state=RANDOM_STATE)
X_d = StandardScaler().fit_transform(X_d)

fig, axes = plt.subplots(2, 4, figsize=(17, 7))
for j, lk in enumerate(["single", "complete", "average", "ward"]):
    Z = scipy_linkage(X_d, method=lk)
    dendrogram(Z, ax=axes[0, j], no_labels=True, color_threshold=Z[-2, 2])
    axes[0, j].set_title(lk + ("  <- ваш вариант" if lk == variant["linkage"] else ""),
                         fontsize=10)
    lab = fcluster(Z, 3, criterion="maxclust") - 1
    axes[1, j].scatter(*X_d.T, c=lab, s=14, cmap="viridis")
    axes[1, j].set_title(f"ARI = {adjusted_rand_score(y_d, lab):.3f}", fontsize=9)
    axes[1, j].set_xticks([]); axes[1, j].set_yticks([])
axes[0, 0].set_ylabel("расстояние слияния")
plt.tight_layout(); plt.show()

In [ ]:
# Эффект цепочки: две вытянутые полосы и десяток точек-мостика между ними
g = np.random.default_rng(0)
X_ch = np.vstack([np.column_stack([g.uniform(-3, 3, 90), g.normal(1.6, 0.12, 90)]),
                  np.column_stack([g.uniform(-3, 3, 90), g.normal(-1.6, 0.12, 90)]),
                  np.column_stack([g.normal(0, 0.05, 12), g.uniform(-1.5, 1.5, 12)])])

fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, lk in zip(axes, ["single", "complete", "average", "ward"]):
    lab = fcluster(scipy_linkage(X_ch, method=lk), 2, criterion="maxclust")
    ax.scatter(*X_ch.T, c=lab, s=12, cmap="coolwarm")
    ax.set_title(f"{lk}: размеры кластеров {np.bincount(lab)[1:]}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Эффект цепочки: 12 точек-мостика склеивают две полосы")
plt.tight_layout(); plt.show()

> **Вывод.** Какой linkage пострадал от эффекта цепочки и почему? Когда это свойство оказывается полезным?
>
> *(ваш ответ здесь)*

---
# Часть 6. Своя выборка

Кластеризуем признаковое описание **без** использования целевой переменной
и проверим, связаны ли найденные кластеры с целью. Это типичный разведочный
сценарий: кластеры могут отражать сегменты, которые полезно рассматривать
отдельно.

In [ ]:
data = load_personal(variant, return_frame=True)
Xp = np.vstack([data["X_train"], data["X_test"]])
yp = np.r_[data["y_train"], data["y_test"]]
target_bin = ((yp > np.median(yp)).astype(int) if data["task"] == "regression"
              else yp.astype(int))
print(f"{data['domain']}: {Xp.shape[0]} объектов, {Xp.shape[1]} признаков")

rows = []
for K in range(2, 9):
    km = KMeans(K, n_init=10, random_state=RANDOM_STATE).fit(Xp)
    rows.append({"K": K, "Q": km.inertia_, "силуэт": silhouette_score(Xp, km.labels_),
                 "связь с целью (AMI)": adjusted_mutual_info_score(target_bin, km.labels_)})
own = pd.DataFrame(rows).set_index("K")
display(own.round(4))

In [ ]:
K_best = int(own["силуэт"].idxmax())
km_best = KMeans(K_best, n_init=10, random_state=RANDOM_STATE).fit(Xp)
print(f"выбрано K = {K_best} (по силуэту)\n")

tab = pd.crosstab(pd.Series(km_best.labels_, name="кластер"),
                  pd.Series(target_bin, name=data["target"]), normalize="index")
tab["размер"] = pd.Series(km_best.labels_).value_counts().sort_index()
display(tab.round(3))

centers = pd.DataFrame(km_best.cluster_centers_, columns=data["feature_names"])
spread = (centers.max() - centers.min()).sort_values(ascending=False).head(6)
display(centers[spread.index].round(2).T.rename(columns=lambda k: f"кластер {k}"))

> **Вывод.** Связаны ли найденные кластеры с целевой переменной? Чем они отличаются содержательно и означает ли высокий AMI, что кластеризация «удалась»?
>
> *(ваш ответ здесь)*

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Почему $K$-средних всегда сходится, но не обязательно к глобальному минимуму? Приведите конфигурацию из четырёх точек и $K=2$, где неудачная инициализация даёт неоптимальный ответ.
2. Функционал $Q$ монотонно убывает по $K$. Означает ли это, что «чем больше кластеров, тем лучше»? Как правильно поставить вопрос о выборе $K$?
3. Теорема 7.7 гарантирует, что $\ln L$ не убывает. Гарантирует ли она сходимость к глобальному максимуму? Что делают на практике?
4. Силуэт указал $K=2$, BIC — $K=5$. Ваши действия?

---

**Дома:** откройте `lab08_homework.ipynb` — там две задачи: свой алгоритм Ллойда с k-means++ и свой EM для смеси гауссиан.